# Heart Disease Prediction — Decision Tree Classifier
**Dataset:** UCI Heart Disease (heart.csv)  
**Model:** Decision Tree (Gini criterion)  
**Goal:** Predict whether a patient has heart disease (binary classification)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Libraries loaded ✅')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('heart.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print('Null values:\n', df.isnull().sum())
print('\nTarget distribution:\n', df['target'].value_counts())

In [ ]:
# Target distribution plot
sns.countplot(x='target', data=df, palette='Set2')
plt.title('Target Distribution (0 = No Disease, 1 = Heart Disease)')
plt.xticks([0, 1], ['No Disease', 'Heart Disease'])
plt.show()

## 2. Data Cleaning

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('After dedup:', df.shape)

## 3. Outlier Detection

In [ ]:
cols = df.columns.tolist()
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(cols):
    axes[i].boxplot(df[col])
    axes[i].set_title(col)
for j in range(len(cols), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.suptitle('Boxplots — Outlier Detection', y=1.02, fontsize=14)
plt.show()

## 4. Feature Engineering & Split

In [ ]:
x = df.iloc[:, 0:-1]
y = df.iloc[:, -1]
feature_names = list(x.columns)
print('Features:', feature_names)
print('X shape:', x.shape, '| y shape:', y.shape)

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, random_state=42, test_size=0.2
)
print(f'Train size: {x_train.shape[0]} | Test size: {x_test.shape[0]}')

## 5. Model Training

In [ ]:
dt = DecisionTreeClassifier(criterion='gini', random_state=0)
dt.fit(x_train, y_train)
y_predict = dt.predict(x_test)

acc = accuracy_score(y_test, y_predict)
print(f'✅ Accuracy: {acc:.4f} ({acc*100:.2f}%)')

## 6. Evaluation

In [ ]:
print('Classification Report:')
print(classification_report(y_test, y_predict, target_names=['No Disease', 'Heart Disease']))

In [ ]:
cm = confusion_matrix(y_test, y_predict)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Heart Disease'],
            yticklabels=['No Disease', 'Heart Disease'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(dt.feature_importances_, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importances')
plt.tight_layout()
plt.show()

## 7. Save Model (.pkl)

In [ ]:
with open('Heart_Disease.pkl', 'wb') as f:
    pickle.dump({'model': dt, 'feature_names': feature_names}, f)
print('✅ Heart_Disease.pkl saved!')

## 8. Export Tree to JSON

In [ ]:
def tree_to_json(model, feature_names):
    tree = model.tree_
    def recurse(node):
        if tree.feature[node] != _tree.TREE_UNDEFINED:
            return {
                'feature': feature_names[tree.feature[node]],
                'threshold': float(tree.threshold[node]),
                'left': recurse(tree.children_left[node]),
                'right': recurse(tree.children_right[node])
            }
        values = tree.value[node][0].tolist()
        total = sum(values)
        return {
            'value': int(tree.value[node].argmax()),
            'samples': int(tree.n_node_samples[node]),
            'distribution': [round(v, 2) for v in values],
            'confidence': round(max(values) / total, 4) if total > 0 else 1.0
        }
    return {
        'type': 'decision_tree',
        'features': feature_names,
        'classes': ['No Disease', 'Heart Disease'],
        'accuracy': round(float(acc), 4),
        'n_features': len(feature_names),
        'n_classes': 2,
        'tree': recurse(0)
    }

model_json = tree_to_json(dt, feature_names)
with open('Heart.json', 'w') as f:
    json.dump(model_json, f, indent=2)
print('✅ Heart.json saved!')